In [6]:
# ============================================================
# ALLANAI.CrimeVision — UK Crime Dataset Builder (Subset: Jan 2022)
# Author: Mahira
# Source: https://data.police.uk/api/
# ============================================================

import os
import time
import json
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# --- Paths ---
ROOT = Path("data_uk_2023_01")
ROOT.mkdir(parents=True, exist_ok=True)

# --- API Base ---
BASE_URL = "https://data.police.uk/api"

# ============================================================
# 1️ Helper functions
# ============================================================

def get_json(url, params=None):
    """Generic function to get JSON safely."""
    try:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"[ERR] Failed request: {url} | {e}")
        return None

def list_force_neighbourhoods(force_id):
    return get_json(f"{BASE_URL}/{force_id}/neighbourhoods")

def get_neighbourhood_boundary(force_id, neigh_id):
    return get_json(f"{BASE_URL}/{force_id}/{neigh_id}/boundary")

def fetch_crimes_by_centroid(force_id, neigh_id, boundary_points, month):
    """Fetch crimes using centroid of neighbourhood (safe URL length)."""
    if not boundary_points:
        return pd.DataFrame()

    try:
        lat = sum(float(p['latitude']) for p in boundary_points) / len(boundary_points)
        lng = sum(float(p['longitude']) for p in boundary_points) / len(boundary_points)
    except Exception:
        return pd.DataFrame()

    params = {"lat": lat, "lng": lng, "date": month}
    url = f"{BASE_URL}/crimes-street/all-crime"
    data = get_json(url, params=params)
    if not data:
        return pd.DataFrame()

    df = pd.json_normalize(data)
    cols = {
        "id": "crime_id",
        "category": "crime_type",
        "month": "month",
        "location.latitude": "latitude",
        "location.longitude": "longitude",
        "location.street.name": "street_name",
        "outcome_status.category": "outcome"
    }
    for c in cols.keys():
        if c not in df.columns:
            df[c] = pd.NA
    df = df[list(cols.keys())].rename(columns=cols)
    df["force_id"] = force_id
    df["neighbourhood_id"] = neigh_id
    return df

# ============================================================
# 2️ Define subset forces and target month
# ============================================================

forces_subset = [
    # "metropolitan",          # London
    # "greater-manchester",    # Manchester
    # "west-midlands",         # Birmingham
    # "west-yorkshire",        # Leeds
    "thames-valley"          # Oxford / Reading region
]

month = "2023-01"

# ============================================================
# 3️ Main fetching loop (safe version)
# ============================================================

def fetch_force_month(force_id, month, sleep=1.5):
    """Fetch all neighbourhoods for one force and one month."""
    neighs = list_force_neighbourhoods(force_id)
    if not neighs:
        print(f"[WARN] No neighbourhoods found for {force_id}")
        return

    all_dfs = []
    for n in tqdm(neighs, desc=f"{force_id}", leave=False):
        try:
            boundary = get_neighbourhood_boundary(force_id, n["id"])
            if not boundary:
                continue
            df = fetch_crimes_by_centroid(force_id, n["id"], boundary, month)
            if not df.empty:
                all_dfs.append(df)
            time.sleep(sleep)
        except Exception as e:
            print(f"[WARN] {force_id}-{n['id']} {month}: {e}")
            time.sleep(sleep)
            continue

    if all_dfs:
        df_force = pd.concat(all_dfs, ignore_index=True)
        out_path = ROOT / f"{force_id}_{month}.csv"
        df_force.to_csv(out_path, index=False)
        print(f"✅ Saved: {out_path} ({len(df_force)} rows)")
    else:
        print(f"[INFO] No data found for {force_id} in {month}")

# ============================================================
# 4️ Run the subset
# ============================================================

for fid in tqdm(forces_subset, desc=f"Fetching crimes for {month}"):
    fetch_force_month(fid, month)
    time.sleep(3)  # pause between forces

print("\n🎉 Completed subset data collection for January 2022!")
print(f"CSV files saved in: {ROOT.resolve()}")


Fetching crimes for 2023-01:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Saved: data_uk_2023_01/thames-valley_2023-01.csv (15514 rows)


Fetching crimes for 2023-01: 100%|██████████| 1/1 [04:02<00:00, 242.54s/it]


🎉 Completed subset data collection for January 2022!
CSV files saved in: /Users/mahirabanu/Documents/ALLANAI-CrimeVision/notebooks/data_uk_2023_01


In [4]:
# ============================================================
# Combine ALL monthly data into one master dataset
# ============================================================

import pandas as pd
from pathlib import Path

# Paths for each folder
months = ["data_uk_2022_10", "data_uk_2022_11", "data_uk_2022_12"]

# Collect all CSV paths
csv_files = []
for m in months:
    folder = Path(m)
    csv_files.extend(list(folder.glob("*.csv")))

print(f"Found {len(csv_files)} CSV files")

# Read and combine
dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    df["source_file"] = f.name
    dfs.append(df)

# Combine into one master dataset
master_df = pd.concat(dfs, ignore_index=True)

# Drop duplicates if any
master_df = master_df.drop_duplicates(subset=["crime_id"], keep="first")

# Save combined file
out_path = Path("merged/UK_Crime_Oct_Dec_2022.csv")
out_path.parent.mkdir(exist_ok=True, parents=True)
master_df.to_csv(out_path, index=False)

print(f"✅ Merged dataset saved at: {out_path}")
print(f"Rows: {len(master_df):,}, Columns: {len(master_df.columns)}")


Found 15 CSV files
✅ Merged dataset saved at: merged/UK_Crime_Oct_Dec_2022.csv
Rows: 414,382, Columns: 10


In [5]:
master_df.info()
master_df.describe()
master_df['crime_type'].value_counts().head(10)
master_df['month'].value_counts()


<class 'pandas.core.frame.DataFrame'>
Index: 414382 entries, 0 to 1714128
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   crime_id          414382 non-null  int64  
 1   crime_type        414382 non-null  object 
 2   month             414382 non-null  object 
 3   latitude          414382 non-null  float64
 4   longitude         414382 non-null  float64
 5   street_name       414382 non-null  object 
 6   outcome           340249 non-null  object 
 7   force_id          414382 non-null  object 
 8   neighbourhood_id  414382 non-null  object 
 9   source_file       414382 non-null  object 
dtypes: float64(2), int64(1), object(7)
memory usage: 34.8+ MB


month
2022-10    158305
2022-11    133453
2022-12    122624
Name: count, dtype: int64